
# **FACULTAD DE INGENIERÍA - BIOINGENIERÍA**
# **Bioseñales y Sistemas**
## _**Segundo Proyecto - Clasificación de Datos BCI**_

*Maria José Rios Hurtado*

*Mónica Alejandra Hinestroza Chaparro*


### **1. CONSULTA**

**Consultar qué índices del EEG, adicionales a los realizados en el proyecto 1 basados en densidad espectral de potencia, mejoran la clasificación de estados en BCI. Citar las fuentes.**

**Escoger por lo menos tres índices que se puedan implementar en Python, se pueden usar librerías, explicar la medida y su uso en los datos del proyecto.**

<div align = "justify">

La densidad espectral de potencia (PSD) es uno de los métodos más empleados para caracterizar señales EEG en sistemas BCI basados en imaginería motora, pero su capacidad discriminativa tiene límites: opera por canal de forma independiente, asume cierta estacionariedad y no captura la distribución espacial ni la complejidad temporal de la actividad cortical. La literatura reciente identifica varios índices complementarios que mejoran la precisión de clasificación cuando se combinan o reemplazan a la PSD.


En el Proyecto 1 se caracterizó estados motores (reposo, movimiento real e imaginación) mediante la densidad espectral de potencia (PSD) en las bandas mu (8–13 Hz) y beta (13–30 Hz) sobre los canales C3 y C4. Si bien la PSD captura información sobre la energía por frecuencia, no refleja la distribución espacial de la actividad cortical, la dinámica temporal de la señal ni su complejidad no lineal. Los tres índices presentados a continuación complementan directamente los hallazgos de ese proyecto, son ampliamente respaldados por la literatura BCI y son implementables en Python.

</div>

<div align ="justify">

#### **Common Spatial Pattern (CSP)**

El CSP es una técnica de filtrado espacial que mejora la capacidad discriminativa de las señales EEG al ser particularmente útil para problemas de clasificación binaria, como distinguir entre imaginación de movimiento de mano derecha e izquierda. El CSP opera encontrando filtros espaciales que maximizan la varianza para una clase mientras la minimizan para la otra. Este principio lo hace directamente complementario a los análisis PSD del Proyecto 1, donde la lateralización entre C3 y C4 fue el hallazgo central: mientras la PSD cuantifica la potencia en cada canal por separado, el CSP encuentra la combinación lineal de todos los canales que maximiza esa diferencia inter-hemisférica de forma óptima.[1]

Durante la imaginería motora, los ritmos sensoriomotores se atenúan y luego se amplifican en corto tiempo, fenómeno conocido como desincronización/sincronización relacionada con eventos (ERD/ERS). Para extraer de forma óptima las características EEG que describen este fenómeno, el algoritmo CSP busca filtros espaciales que extraigan características espaciales discriminativas entre clases y es adoptado frecuentemente debido a su buen desempeño. Sin embargo, el desempeño del CSP depende fuertemente de la selección de bandas de frecuencia para la extracción del ritmo sensoriomotor. Por ello, variantes como Filter Bank CSP (FBCSP) aplican el método sobre múltiples bandas de frecuencia antes de seleccionar las más discriminativas. La implementación en Python se realiza con mne.decoding.CSP de la librería MNE. [2]

**- Funcionamiento Matemático:** El método calcula matrices de covarianza para cada clase (por ejemplo MI_R y MI_L) y resuelve un problema de autovalores generalizados:

$$\boxed{C_1 ​W = \lambda C_2 W}$$

donde:
 - C1 y C2 representan las matrices de covarianza de las dos clases
 - W corresponde a los filtros espaciales
 - $\lambda$ representa la capacidad discriminativa del filtro

Los filtros obtenidos generan nuevas señales espaciales donde las diferencias entre clases son más evidentes.

**- Interpretación Fisiológica:** En tareas de imaginación motora:

- El canal C3 suele activarse durante imaginación de movimiento de la mano derecha
- El canal C4 suele activarse durante imaginación de movimiento de la mano izquierda

El CSP amplifica estas diferencias espaciales, permitiendo identificar patrones de lateralización cerebral.

**- Implementación en Python:** Se utiliza la librería MNE

    from mne.decoding import CSP

    csp = CSP(n_components=4)
    X_csp = csp.fit_transform(X, y)

**- Ventajas:**

- Alta capacidad discriminativa
- Muy eficiente en BCI motoras
- Reduce dimensionalidad
- Mejora precisión de clasificadores

#### **Parámetros de Hjorth**

Los parámetros de Hjorth son características basadas en la varianza de las derivadas de la señal EEG. Las tres primeras derivadas de la señal dan lugar a las medidas de actividad (Activity), movilidad (Mobility) y complejidad (Complexity), que son los parámetros de Hjorth frecuentemente empleados en sistemas BCI. Estos tres descriptores operan enteramente en el dominio del tiempo, lo que los hace ortogonales a la PSD y muy rápidos de calcular. [3]

Activity mide la varianza cuadrática de la amplitud de la señal EEG en una ventana considerada. Mobility se define como la potencia promedio de la derivada normalizada de la señal EEG. Complexity representa la relación entre las movilidades, es decir, la movilidad de la primera derivada dividida entre la movilidad de la señal original. En términos fisiológicos, Activity es análoga a la potencia total integrada de la PSD, Mobility refleja desplazamientos en la frecuencia media dominante del ritmo mu o beta, y Complexity captura cuánto se aleja la señal de una oscilación sinusoidal pura, lo que aumenta cuando la corteza motora se activa y la señal se vuelve más irregular. [4]

Según resultados recientes, la actividad, movilidad y complejidad de segmentos EEG captados por los canales C3, Cz y C4, con y sin descomposición en sub-bandas, proporcionaron precisiones de clasificación confiables. El enfoque propuesto permitió discriminar tareas de movimiento ejecutado e imaginado, alcanzando una precisión promedio de clasificación de 89.7 ± 0.78%. En estudios comparativos de extracción de características para BCI, se ha reportado que la combinación de potencia de bandas, parámetros de Hjorth y coeficientes autorregresivos adaptativos presenta resultados consistentes en múltiples datasets EEG de BCI. Los parámetros de Hjorth se implementan directamente con NumPy a partir de las derivadas discretas de la señal, sin necesidad de librerías especializadas. [5]

#### **Energía y Entropía Wavelet (DWT)**

La Transformada Wavelet Discreta (DWT) ofrece una representación tiempo-frecuencia de resolución múltiple que supera a la PSD convencional en señales no estacionarias como el EEG. Se propone un método de extracción de características basado en la transformada wavelet discreta (DWT), la descomposición en modos empíricos (EMD) y la entropía aproximada. La señal EEG se descompone en una serie de señales de banda estrecha con la DWT, y la entropía aproximada de la señal reconstruida se obtiene como el vector de características correspondiente. [5]

De la DWT se derivan dos índices especialmente útiles para BCI. El primero es la energía relativa por sub-banda, que representa la fracción de la energía total contenida en cada nivel wavelet, siendo directamente comparable con la PSD integrada por banda del Proyecto 1 pero con mejor localización temporal. El segundo es la entropía de Shannon wavelet, que mide qué tan distribuida está la energía entre las sub-bandas: la entropía wavelet relativa y las características topológicas de la red de función cerebral pueden extraerse de forma conjunta, logrando una precisión de clasificación promedio superior al 90% en datasets públicos de BCI, lo que demuestra que estas características retienen información de la señal EEG de forma más efectiva y reducen la complejidad computacional. [5]

En el proceso de extracción de características es posible calcular energía, varianza y entropía de la transformada wavelet basadas en cinco sub-bandas de frecuencia EEG como características en el dominio tiempo-frecuencia, las cuales, combinadas con parámetros en el dominio del tiempo y del dominio de la frecuencia, permiten obtener los mejores resultados de selección en clasificaciones binarias y multiclase de imaginería motora. La implementación en Python se realiza con la librería PyWavelets (pywt), que permite la descomposición DWT con wavelets como db4, cuyos niveles se alinean con las bandas delta, theta, mu y beta para una frecuencia de muestreo de 160 Hz.[6]

#### **Coherencia**
El análisis de coherencia en el electroencefalograma (EEG) es una métrica neurofisiológica que evalúa la conectividad funcional y la sincronización entre distintas regiones del cerebro. Actúa como un índice que mide qué tan coordinadamente se comunican y oscilan dos áreas corticales a través de bandas de frecuencia específicas.

La coherencia es una medida de conectividad funcional que cuantifica el grado de sincronización entre dos señales EEG en una banda de frecuencia específica.Esta métrica permite evaluar cómo interactúan distintas regiones cerebrales durante tareas cognitivas o motoras.[7]

La coherencia toma valores entre:

0 → sin relación
1 → sincronización perfecta

**- Funcionamiento Matemático:** La coherencia entre dos señales x(t) e y(t) se define como:

$$\boxed{C_{xy}(f) = \frac{|P_{xy}(f)|^2}{P_{xx}(f) \cdot P_{yy}(f)}}$$

donde:

- P_xy(f): densidad espectral cruzada
- P_yy(f), P_xx(f) : densidades espectrales de potencia individuales

**- Interpretación Fisiológica:** En tareas motoras:

Aumentos de coherencia pueden indicar comunicación funcional entre áreas motoras
Disminuciones pueden reflejar especialización o lateralización de la actividad

La coherencia es especialmente útil para estudiar:

conectividad cortical
coordinación interhemisférica
organización funcional cerebral

**- Implementación en Python:** Se utilizan las librerias MNE y SciPy

    from scipy.signal import coherence

    f, coh = coherence(signal1, signal2, fs=fs)

**. Ventajas**

- Evalúa interacción entre regiones cerebrales
- Complementa análisis espectral
- Permite estudiar redes neuronales funcionales

#### **Entropía Espectral**

La entropía espectral en el Electroencefalograma (EEG) es una métrica de la teoría de la información que cuantifica la irregularidad o complejidad de las señales cerebrales. Se calcula a partir de la distribución de potencia en diferentes frecuencias; valores altos indican un cerebro despierto y activo, mientras que valores bajos reflejan regularidad y menor actividad.

La entropía espectral es una medida de complejidad basada en la distribución de energía de la señal EEG en el dominio de la frecuencia.Esta métrica cuantifica el grado de desorganización o irregularidad de la actividad cerebral.

Valores bajos → actividad más regular y organizada
Valores altos → actividad más compleja y distribuida

**- Fundamento Matemático:** La entropía espectral se calcula aplicando la entropía de Shannon sobre la PSD normalizada:

$$\boxed{H = -\sum_{i=1}^{N}p_i \log(p_i)}$$

donde:

- pi: representa la distribución normalizada de potencia espectral
- H: corresponde a la entropía espectral

**- Interpretación Fisiológica:** En EEG

Estados de reposo suelen presentar patrones más organizados
Tareas cognitivas o motoras incrementan la complejidad neuronal

La entropía espectral permite detectar cambios dinámicos en la organización cortical

**- Implementación en Python:** Se utilizan las librerias AntroPy y NumPy

    import antropy as ant

    entropy = ant.spectral_entropy(signal, sf=fs)

**. Ventajas**

- Captura complejidad neuronal
- Sensible a cambios cognitivos
- Complementa PSD y conectividad

#### **Selección de Índices**

De los índices consultados, se seleccionan la coherencia espectral y la entropía espectral como características escalares por canal y por época, las cuales se integrarán directamente en el DataFrame de análisis junto con la PSD del Proyecto 1. Estos dos índices son complementarios entre sí y con la PSD: mientras la PSD cuantifica la energía por banda en cada canal de forma independiente, la coherencia aporta información sobre la sincronización funcional entre regiones cerebrales, y la entropía espectral captura la complejidad e irregularidad de la actividad cortical. En conjunto, ofrecen una representación más completa de la dinámica cerebral durante las tareas de imaginería motora.

El CSP también se incluye en el proyecto, pero con un rol diferente al de los índices anteriores. A diferencia de la coherencia y la entropía, el CSP no produce valores escalares independientes por canal sino filtros espaciales que transforman conjuntamente todos los canales, y su ajuste requiere conocer las etiquetas de clase. Por estas razones, su lugar natural dentro del pipeline no es la extracción de características previa al DataFrame, sino la etapa de clasificación, donde se aplica como paso de preprocesamiento supervisado antes de los clasificadores, tal como es su uso estándar en la literatura BCI [1, 2]. Esta distinción se detalla en el Plan de Análisis.

</div>

### **2. PLAN DE ANÁLISIS**

**Con la información recolectada, del entregable 1 y del punto anterior, proponer una metodología de análisis que permita evidenciar la diferencia en ritmos cerebrales asociados a las condiciones de: reposo, imaginación de movimiento mano derecha, imaginación de movimiento mano izquierda.** 

Con base en el análisis realizado en el Proyecto 1 y en los índices adicionales consultados, se propone la siguiente metodología para caracterizar y comparar la actividad cerebral asociada a tres condiciones experimentales: reposo (REST), imaginación de movimiento de la mano derecha (MI_R) e imaginación de movimiento de la mano izquierda (MI_L).
La metodología sigue el flujo recomendado para el desarrollo de sistemas BCI [diapositivas de clase], que va desde la preparación de los datos hasta la evaluación del modelo de clasificación.

_**Etapa 1 — Preparación de datos y segmentación**_

Se reutiliza el pipeline de preprocesamiento del Proyecto 1 (filtro pasa-banda 1–40 Hz, filtro notch 60 Hz, rereferenciación CAR) aplicado sobre los canales C3, C4 y Cz. A partir de la señal continua preprocesada, se extraen épocas delimitadas por los marcadores de evento del dataset: T0 (reposo), T1 (MI_L) y T2 (MI_R), con una ventana de [0 s, 4 s] respecto al onset del estímulo. Esta segmentación ya fue implementada en el Proyecto 1 y se reutiliza directamente.

_**Etapa 2 — Extracción de características**_

Para cada época y cada canal de interés se calculan los siguientes índices, que conformarán las columnas del DataFrame de características:

a) Potencia espectral por bandas (PSD — mu y beta): heredada directamente del Proyecto 1. Calcula la potencia media en las bandas mu (8–13 Hz) y beta (13–30 Hz) usando el método de Welch. Produce un valor escalar por banda y por canal (ej. psd_mu_C3, psd_beta_C4).

b) Coherencia espectral (C3–C4, C3–Cz, C4–Cz): mide el grado de sincronización funcional entre pares de canales en las bandas mu y beta. Se calcula con scipy.signal.coherence y se promedia dentro de cada banda. Produce un valor escalar por par de canales y por banda (ej. coh_mu_C3C4, coh_beta_C3Cz).

c) Entropía espectral: cuantifica la complejidad de la señal midiendo qué tan distribuida está la energía espectral entre las frecuencias. Se calcula con antropy.spectral_entropy. Produce un valor escalar por canal (ej. ent_C3, ent_C4).

Todos estos índices se almacenan en un único DataFrame donde cada fila representa una época (un registro), con columnas para el sujeto, la condición y cada métrica calculada por canal.

_**Sobre el CSP:** el Common Spatial Pattern no se incluye como característica en el DataFrame porque su naturaleza es fundamentalmente diferente a los índices anteriores. El CSP no produce valores escalares independientes por canal, sino filtros espaciales que transforman conjuntamente todos los canales. Su entrenamiento requiere las etiquetas de clase, lo que lo hace parte del pipeline de clasificación (no de extracción de características previa). El CSP se aplicará por tanto en la Etapa 5, como paso de preprocesamiento supervisado antes del clasificador, tal como se hace habitualmente en la literatura BCI [1, 2]._

_**Etapa 3 — Organización del DataFrame y preparación para clasificación**_

El DataFrame resultante tendrá la siguiente estructura, consistente con el formato solicitado en el enunciado:

| Registro | Tarea | psd_mu_C3 | psd_beta_C4 | coh_mu_C3C4 | ent_C3 | ... |
|----------|-------|-----------|-------------|-------------|--------|-----|
| S001_R04_epoch0 | MI_R | ... | ... | ... | ... | ... |

Una vez construido el DataFrame se realizan los siguientes pasos de preparación antes de alimentar los clasificadores:

- _Codificación de etiquetas:_ las etiquetas textuales (REST, MI_R, MI_L) se convierten a valores numéricos (0, 1, 2).
- _Normalización de características:_ todas las columnas de características se normalizan al mismo rango para evitar que índices de mayor magnitud dominen el entrenamiento. La etiqueta no se normaliza.
- _Análisis de correlación:_ se calcula la matriz de correlación entre características. Correlaciones superiores a 0.85 en valor absoluto son señal de redundancia que puede perjudicar el modelo; en ese caso se elimina una de las características correlacionadas.
- _Balance de clases:_ se verifica que las tres condiciones tengan representación proporcional en el conjunto de datos, dado que los clasificadores son sensibles al desbalance de clases.
- _División train/test:_ se separa el 80% de los datos para entrenamiento y el 20% para evaluación, con random_state fijo para garantizar reproducibilidad.

_**Etapa 4 — Análisis estadístico descriptivo e inferencial**_

Sobre el DataFrame de características se realizan dos tipos de análisis:

a) _Estadística descriptiva:_ se calculan media, mediana, desviación estándar e IQR para cada índice por condición. Las visualizaciones se seleccionan según el tipo de comparación: boxplots para comparar distribuciones entre condiciones, mapas de calor para la matriz de correlación entre características, y matrices de conectividad para los valores de coherencia entre pares de canales.

b) _Pruebas de hipótesis:_ se plantean hipótesis formales para los índices que visualmente muestren mayor diferencia entre condiciones. La elección entre prueba paramétrica o no paramétrica depende del resultado de la prueba de normalidad (Shapiro-Wilk). Para comparaciones entre más de dos grupos se aplicará ANOVA o Kruskal-Wallis; para comparaciones entre pares de condiciones, t-test o Wilcoxon. En todos los casos:

- **H₀:** no existen diferencias significativas entre las condiciones comparadas en el índice evaluado.
- **H₁:** existen diferencias significativas entre las condiciones comparadas en el índice evaluado.

El umbral de significancia es α = 0.05.

_**Etapa 5 — Clasificación**_

La clasificación se realiza usando únicamente los índices que hayan mostrado mayor diferencia estadística entre condiciones en la Etapa 4. El pipeline de clasificación sigue el esquema enseñado en clase:

1. _Preprocesamiento con CSP:_ antes de alimentar el clasificador, se aplica CSP sobre las épocas para extraer componentes espaciales discriminativos entre clases. Esto se realiza con mne.decoding.CSP dentro del pipeline de entrenamiento, garantizando que el CSP se ajuste únicamente con datos de entrenamiento en cada fold para evitar fuga de información.

2. _Estrategia de validación cruzada (k-fold):_ se utiliza K-Fold con k=10 para estimar el desempeño del modelo de forma robusta, evitando tanto overfitting como underfitting. El score final reportado es el promedio de las k iteraciones.

3. _Arquitecturas de clasificación:_ se entrenan y comparan al menos tres configuraciones de red neuronal MLP (MLPClassifier) con diferentes números de capas y neuronas, evaluando su desempeño con el classification_report tanto en entrenamiento como en evaluación.

4. _Modelos adicionales:_ se implementan y discuten una Máquina de Soporte Vectorial (SVM) y Extreme Gradient Boosting (XGBoost), comparando su desempeño con las redes MLP.

5. _Métricas de evaluación:_ para cada modelo se reportan la matriz de confusión (normalizada y sin normalizar), accuracy, precisión, recall y F1-score. Se discuten los falsos positivos y negativos en el contexto clínico de un sistema BCI.

### **2. PLAN DE ANÁLISIS**

**Con la información recolectada, del entregable 1 y del punto anterior, proponer una metodología de análisis que permita evidenciar la diferencia en ritmos cerebrales asociados a las condiciones de: reposo, imaginación de movimiento mano derecha, imaginación de movimiento mano izquierda.** 

Con base en el análisis realizado en el entregable anterior y en los índices adicionales consultados, se propone la siguiente metodología para identificar diferencias en la actividad cerebral asociada a:

- Reposo
- Imaginación de movimiento mano derecha (MI_R)
- Imaginación de movimiento mano izquierda (MI_L)

**1. Segmentación de señales (Epoching)**

Las señales EEG se segmentarán utilizando los marcadores de eventos del dataset:

- MI_R
- MI_L
- Reposo

Cada época incluirá:

- Ventana pre-estímulo
- Ventana activa

Esto permitirá comparar actividad cerebral entre condiciones específicas.

**2. Extracción de características** 

Se calcularán múltiples índices EEG:

- **Common Spatial Patterns (CSP)**

_Objetivo:_ Maximizar discriminación entre MI_R y MI_L.

Resultado esperado:patrones espaciales diferenciados entre hemisferios.

- **Coherencia**

_Objetivo:_ Evaluar conectividad funcional entre regiones motoras.

Canales:

- C3–C4
- C3–Cz
- C4–Cz

- **Entropía espectral**

_Objetivo:_ Analizar complejidad cortical entre condiciones.


**3. Análisis Estadístico** 

Se calcularán:

- Media
- Mediana
- Desviación estándar
- IQR

_Visualizaciones:_

- Boxplots
- Mapas de calor
- Matrices de conectividad

Esta es la etapa central de la metodología propuesta. Se calculan los tres índices sobre cada época y canal, construyendo un vector de características por época que combina información espectral (heredada del Proyecto 1), espacial, temporal y tiempo-frecuencial.

**4. Pruebas de hipótesis**

- **Hipótesis principales**

H0: No existen diferencias significativas entre reposo, MI_R y MI_L en los índices EEG calculados.

H1: Existen diferencias significativas entre las condiciones, especialmente en ritmos mu y patrones espaciales motores.

- **Pruebas Estadísticas** 

Dependiendo de la normalidad se aplicará:

- ANOVA de medidas repetidas
- t-test pareado
- Wilcoxon
- Kruskal-Wallis

**PROYECTO - MANEJO DEL DATAFRAME**

<div align =="justify">

Para el proyecto, cambiar las etiquetas en el Datframe de "Mano derecha" a valores numericos.

Se normalizan las caracteristicas, no influye la etiqueta, se normalizan todas las caracteristicas para que queden en un solo rango.

Se debe bajar el valor de la correlación entre las carcteristicas lo ideal es que sean casi nulas. Realizar el análisis de correlación a alta correlación puede ser motivo de deficiencia del modelo.

El resultado de la normalización de los valores de características es el array _x_ y el _y_ es el de las etiquetas.

Se deben dividir los datos en 80-20 , 80% de los datos para entrenar el modelo y el 20% pra evaluarlo o validarlo.

Con train_test_split obtenemos cuatro vectores, los datos de entrenamiento, validación, etiquetas y...

La **matriz de confusión** permite identificar cuántos datos clasifica de forma correcta. Es decir, da información de los falsos negativos y positivos, aquellos valores que pertenecían a una clase pero el modelo los calificó como si fueran de la otra clase.

Especificidad y sensibilidad, para decidir ante los resultados que de el modelo. 

Es importane que el modelo reciba datos proporcionales, igual cantidad de datos de cada clase.

**Clasification_report:** Compara las etiquetas predichas con las reales. Se hace para entrenamiento y evaluación 

</div>

### **3. PROGRAMACIÓN**

**a. Crear una función que reciba una señal de EEG procesada y permita obtener la densidad espectral de potencia y los índices adicionales consultados en un conjunto de canales de interés.**

**b. Crear una rutina que aplique sobre todos los archivos de la base de datos la rutina de procesamiento y la de extracción de caracteríticas y almacene los resultados en un dataframe donde se pueda discriminar nombre sujeto, la condición evaluada y la métrica calculada.**

In [1]:
# IMPORTACIONES NECESARIAS

%matplotlib inline
import mne
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import os
import warnings
import pandas as pd
from scipy import stats
from scipy.signal import coherence
import antropy as ant

warnings.filterwarnings('ignore')

In [ ]:
# CONFIGURACIÓN GLOBAL

# RUNS_TAREA: solo runs de imaginación motora (MI).
#   Run 4, 8, 12 → T1 = MI_L (img izq), T2 = MI_R (img der)
#   T0 en cualquier run → REST (reposo entre tareas)

RUTA_DATOS = r'C:\Users\mrios\Desktop\Universidad\Semestre VII\Bioseñales\files'

RUNS_TAREA = {
    4:  'MI',
    8:  'MI',
    12: 'MI',
}

CANALES    = ['C3', 'C4', 'Cz']
DUR_EPOCA  = 4.0    # segundos por época

# Mapeo condición textual → etiqueta numérica
ETIQUETAS = {
    'REST': 0,
    'MI_L': 1,
    'MI_R': 2,
}

In [ ]:
# FUNCIONES HEREDADAS DEL PROYECTO 1

def cargar_eeg_con_eventos(ruta_edf):
    """
    Carga un archivo EDF del dataset PhysioNet EEGBCI,
    estandariza los nombres de canales y asigna el montaje 10-20.

    Parámetros
    ----------
    ruta_edf : str
        Ruta al archivo .edf

    Retorna
    -------
    archivo : mne.io.Raw
        Señal cargada, lista para preprocesar.
    """
    archivo = mne.io.read_raw_edf(ruta_edf, preload=True, verbose=False)
    mne.datasets.eegbci.standardize(archivo)
    montage = mne.channels.make_standard_montage('standard_1020')
    archivo.set_montage(montage, verbose=False)
    return archivo


def procesar_eeg(archivo, graficar=False):
    """
    Aplica el pipeline de preprocesamiento del Proyecto 1:
      1. Filtro pasa-banda (1–40 Hz)
      2. Filtro notch (60 Hz)
      3. Rereferenciación CAR (Common Average Reference)

    Parámetros
    ----------
    archivo : mne.io.Raw
        Señal cruda cargada con cargar_eeg_con_eventos().
    graficar : bool
        En el pipeline masivo se mantiene en False para no
        generar cientos de figuras.

    Retorna
    -------
    archivo_proc : mne.io.Raw
        Señal preprocesada, lista para extracción de características.
    """
    archivo_proc = (archivo
                    .copy()
                    .filter(l_freq=1.0, h_freq=40.0, verbose=False)
                    .notch_filter(freqs=60.0, verbose=False)
                    .set_eeg_reference('average', verbose=False))
    return archivo_proc


def get_condicion(tipo_run, anotacion):
    """
    Traduce (tipo_run, anotación T0/T1/T2) a etiqueta de condición.

    Retorna
    -------
    str : 'REST' | 'MI_L' | 'MI_R' | 'MM_L' | 'MM_R' | 'UNKNOWN'
    """
    if anotacion == 'T0':
        return 'REST'
    elif anotacion == 'T1':
        return f'{tipo_run}_L'
    elif anotacion == 'T2':
        return f'{tipo_run}_R'
    else:
        return 'UNKNOWN'

In [4]:
# FUNCIÓN DE EXTRACCIÓN DE CARACTERÍSTICAS

# Recibe los datos crudos de una época (array numpy) ya preprocesada y calcula tres grupos de índices:

#   1. PSD por bandas (mu, beta)
#      Heredada del Proyecto 1. Cuantifica la energía espectral en las bandas motoras relevantes por canal.

#   2. Coherencia espectral por pares de canales (mu, beta)
#      Mide la sincronización funcional entre regiones motoras.
#      Pares: C3-C4 (interhemisférica), C3-Cz, C4-Cz.

#   3. Entropía espectral por canal
#      Cuantifica la complejidad de la distribución espectral.
#      Valores altos → actividad distribuida (tarea activa).
#      Valores bajos → actividad concentrada (reposo, ritmo mu).

def extraer_caracteristicas(data_epoca, sfreq, canales_usados):
    """
    Calcula PSD por bandas, coherencia espectral y entropía
    espectral sobre una época de señal EEG preprocesada.

    Parámetros
    ----------
    data_epoca : np.ndarray, shape (n_canales, n_muestras)
        Datos de la época en µV. Las filas deben estar en el
        mismo orden que canales_usados.
    sfreq : float
        Frecuencia de muestreo en Hz.
    canales_usados : list of str
        Nombres de los canales en el mismo orden que las filas
        de data_epoca. Ej: ['C3', 'C4', 'Cz'].

    Retorna
    -------
    resultado : dict
        Diccionario plano con todas las características.
        Claves del tipo:
          'mu_C3', 'mu_C4', 'mu_Cz'         → PSD banda mu
          'beta_C3', 'beta_C4', 'beta_Cz'   → PSD banda beta
          'coh_mu_C3C4', 'coh_mu_C3Cz', ... → coherencia mu
          'coh_beta_C3C4', ...              → coherencia beta
          'ent_C3', 'ent_C4', 'ent_Cz'      → entropía espectral
    """

    resultado = {}
    n_muestras = data_epoca.shape[1]

    # PSD POR BANDAS (mu y beta)

    # Misma metodología que el Proyecto 1: método de Welch con ventana de Hamming, n_fft=256 y solapamiento del 50%.
    # Se calcula el promedio de potencia dentro de cada banda → un escalar por (banda, canal).
    # data_epoca está en µV → psd_array_welch espera V, por eso se divide por 1e6 antes de pasarlo.
    # Al final se multiplica por 1e12 para volver a µV²/Hz.

    bandas = {
        'mu'  : (8.0,  13.0),
        'beta': (13.0, 30.0),
    }

    # n_fft no puede ser mayor que n_muestras de la época
    n_fft = min(256, n_muestras)

    psd_v2, freqs = mne.time_frequency.psd_array_welch(
        data_epoca * 1e-6,      # µV → V
        sfreq     = sfreq,
        fmin      = 1.0,
        fmax      = 40.0,
        n_fft     = n_fft,
        n_overlap = n_fft // 2,
        window    = 'hamming',
        verbose   = False
    )
    psd_uv2 = psd_v2 * 1e12    # V²/Hz → µV²/Hz

    # Vectorizado: psd_uv2[:, mask] → (n_canales, n_bins_banda)
    # .mean(axis=1) → (n_canales,) — sin bucle sobre canales
    for nombre_banda, (fmin_b, fmax_b) in bandas.items():
        mask = (freqs >= fmin_b) & (freqs <= fmax_b)
        if not np.any(mask):
            for ch in canales_usados:
                resultado[f'{nombre_banda}_{ch}'] = np.nan
            continue
        potencias = psd_uv2[:, mask].mean(axis=1)
        for i, ch in enumerate(canales_usados):
            resultado[f'{nombre_banda}_{ch}'] = potencias[i]

    # COHERENCIA ESPECTRAL POR PARES DE CANALES

    # scipy.signal.coherence devuelve Cxy(f) ∈ [0,1] para todo el espectro. Se promedia dentro de cada banda, igual que con la PSD, para obtener un escalar por (par, banda).

    # La coherencia es una medida normalizada → no depende de la escala de la señal, por lo que se puede usar data_epoca en µV directamente sin conversión de unidades.

    # nperseg=n_fft para consistencia con el bloque anterior.

    idx_ch = {ch: i for i, ch in enumerate(canales_usados)}

    # Solo calcular pares cuyos dos canales estén disponibles
    pares = []
    for ch_a, ch_b in [('C3', 'C4'), ('C3', 'Cz'), ('C4', 'Cz')]:
        if ch_a in idx_ch and ch_b in idx_ch:
            pares.append((ch_a, ch_b))

    for ch_a, ch_b in pares:
        f_coh, Cxy = coherence(
            data_epoca[idx_ch[ch_a]],
            data_epoca[idx_ch[ch_b]],
            fs      = sfreq,
            nperseg = n_fft
        )
        for nombre_banda, (fmin_b, fmax_b) in bandas.items():
            mask = (f_coh >= fmin_b) & (f_coh <= fmax_b)
            nombre_col = f'coh_{nombre_banda}_{ch_a}{ch_b}'
            if not np.any(mask):
                resultado[nombre_col] = np.nan
            else:
                resultado[nombre_col] = Cxy[mask].mean()

    # ENTROPÍA ESPECTRAL POR CANAL

    # antropy.spectral_entropy aplica la entropía de Shannon sobre la PSD normalizada de la señal.
    # normalize=True → valores entre 0 y 1, comparables entre sujetos y canales independientemente de la amplitud.
    
    # Un canal en reposo con ritmo mu dominante tendrá entropía baja (energía concentrada en ~10 Hz).
    # Un canal durante tarea motora tendrá entropía más alta (energía distribuida por ERD/ERS en múltiples frecuencias).
  
    for i, ch in enumerate(canales_usados):
        resultado[f'ent_{ch}'] = ant.spectral_entropy(
            data_epoca[i],
            sf        = sfreq,
            method    = 'welch',
            normalize = True
        )

    return resultado

### **4. ANÁLISIS DE RESULTADOS**

**a. Discusión de las diferencias en las condiciones para los diferentes índices usando las gráficas obtenidas usando estadística descriptiva. Seleccione el tipo de representación que mejor le permita describir la información (ver el enlace recomendado) y use comparaciones válidas.**

**b. Planteamiento de las hipótesis nulas y alternativas, selección del tipo de prueba (paramétrica o no paramétrica) y discusión de los resultados.**

### **5. CLASIFICACIÓN**

**Todos los puntos realizados a continuación son con las índices que muestren mayor diferencia entre las condiciones.**

**a. Código y discusión de una estrategia de entrenamiento basada en k-fold.**

**b. Código y análisis de resultados, donde se discutan por los menos tres diferentes arquitecturas de red (5%) y las matrices de confusión obtenidas, para la clasificación de las diferentes condiciones.**

**c. Consultar cómo funciona, realizar y discutir un ejemplo con los datos del algoritmo máquina de soporte vectorial y extreme gradient boosting.**

### **REFERENCIAS**


[1] Blankertz, B., Tomioka, R., Lemm, S., Kawanabe, M., & Müller, K.-R. (2008). Optimizing spatial filters for robust EEG single-trial analysis. IEEE Signal Processing Magazine, 25(1), 41–56. https://doi.org/10.1109/MSP.2008.4408441

[2] Degirmenci, M., Yuce, Y. K., Perc, M., & Isler, Y. (2023). Statistically significant features improve binary and multiple Motor Imagery task predictions from EEGs. Frontiers in Human Neuroscience, 17, 1223307. https://doi.org/10.3389/fnhum.2023.1223307

[3] Elahi, M. H. et al. (2023). EEG-BCI Features Discrimination between Executed and Imagined Movements Based on FastICA, Hjorth Parameters, and SVM. Mathematics, 11(21), 4409. https://doi.org/10.3390/math11214409

[4] Garcia-Laencina, P. J. et al. (2014). Exploring dimensionality reduction of EEG features in motor imagery task classification. Expert Systems with Applications, 41(10), 4761–4771. https://doi.org/10.1016/j.eswa.2014.02.021

[5] Ji, N., Ma, L., Dong, H., & Zhang, X. (2019). EEG Signals Feature Extraction Based on DWT and EMD Combined with Approximate Entropy. Brain Sciences, 9(8), 201. https://doi.org/10.3390/brainsci9080201

[6] Wang, M. et al. (2023). Motor imagery classification method based on relative wavelet packet entropy brain network and improved lasso. Frontiers in Neuroscience, 17, 1113593. https://doi.org/10.3389/fnins.2023.1113593

[7] M. Atienza, J.L. Cantero-Lorente, R.M. Salas. Valor clínico de la coherencia EEG como índice electrofisiológico de conectividad córticocortical durante el sueño. Rev. Neurol. 2000, 31(5), 442-454. https://doi.org/10.33588/rn.3105.99472
 
[8] MNE Documentation: https://mne.tools
